# Train Image Generation Using GAN From Scratch
A beginner-friendly **DCGAN** in PyTorch: train small 64×64 RGB faces without pretrained weights.

- **Generator (G):** turns 100 random numbers into an image.
- **Discriminator (D):** outputs one raw score per image (real versus generated).
- **Batch:** images processed together. **Epoch:** one pass through the training dataset.
- **Loss:** a training signal, not an image-quality rating.

In Colab select **Runtime → Change runtime type → GPU**, then run cells in order.
This notebook starts AFTER extracting your CelebA archive. Keep images in `/content` (fast local storage); save checkpoints to Drive (persistent storage). Local files disappear when the runtime resets. Use CelebA only as permitted by its dataset terms; generated faces are synthetic and not verified identities.


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')


## 1. Imports and settings
Start with 10,000 images to check the workflow. Increase `MAX_IMAGES` to `None` for all images once everything works. Five epochs is an initial experiment, not a promise of realistic faces. CPU training is only practical for small debugging runs.


In [ ]:
from pathlib import Path
import random
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image as save_tensor_image, make_grid
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Your unzip command used "-d .", which normally extracted here, NOT to Drive.
IMAGE_DIR = Path('/content/img_align_celeba/img_align_celeba')
OUTPUT_DIR = Path('/content/gdrive/MyDrive/Image_Generation_using_Gan/dcgan_run')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_IMAGES = 10_000
BATCH_SIZE = 64
LATENT_DIM = 100
EPOCHS = 5  # Total target epoch count, including any resumed epochs.


## 2. Load and inspect real images
CelebA images are not square: resize the shorter edge, then center-crop instead of stretching faces. `ToTensor` maps pixels to [0,1]; normalization maps them to [-1,1], matching the generator's final `Tanh`.
No class labels are needed for this unconditional GAN. This tutorial does not use a test loader or claim a held-out quality metric.


In [ ]:
class FaceImages(Dataset):
    def __init__(self, root, limit=None):
        if not root.is_dir():
            raise FileNotFoundError(f'Image directory missing: {root}. Check the unzip destination.')
        self.files = sorted(root.glob('*.jpg'))
        random.Random(SEED).shuffle(self.files)
        if limit is not None:
            self.files = self.files[:limit]
        if len(self.files) < 2:
            raise ValueError('Need at least two JPG images. Check IMAGE_DIR.')
        self.transform = transforms.Compose([
            transforms.Resize(64),
            transforms.CenterCrop(64),
            transforms.ToTensor(),
            transforms.Normalize((0.5,)*3, (0.5,)*3),
        ])
    def __len__(self):
        return len(self.files)
    def __getitem__(self, index):
        with Image.open(self.files[index]) as im:
            return self.transform(im.convert('RGB'))

dataset = FaceImages(IMAGE_DIR, MAX_IMAGES)
loader = DataLoader(dataset, batch_size=min(BATCH_SIZE, len(dataset)),
                    shuffle=True, num_workers=2, pin_memory=device.type == 'cuda',
                    drop_last=True)
real_batch = next(iter(loader))
print('Images:', len(dataset), '| Batch:', real_batch.shape)
plt.figure(figsize=(8,8))
plt.imshow(make_grid(real_batch[:25], nrow=5, normalize=True, value_range=(-1,1)).permute(1,2,0))
plt.axis('off')
plt.show()


## 3. Define both networks
G grows spatial dimensions: **1 → 4 → 8 → 16 → 32 → 64** using transposed convolutions.
D shrinks them: **64 → 32 → 16 → 8 → 4 → 1** using convolutions.
BatchNorm helps stabilize intermediate activations. D uses LeakyReLU to retain gradients for negative activations.
D has **no Sigmoid** because `BCEWithLogitsLoss` includes it in a numerically stable calculation.


In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(LATENT_DIM, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False), nn.Tanh(),
        )
    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512), nn.LeakyReLU(0.2, True),
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
        )
    def forward(self, images):
        return self.net(images).view(-1)

def initialize_weights(layer):
    if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(layer.weight, 0.0, 0.02)
    elif isinstance(layer, nn.BatchNorm2d):
        nn.init.normal_(layer.weight, 1.0, 0.02)
        nn.init.zeros_(layer.bias)

G, D = Generator().to(device), Discriminator().to(device)
G.apply(initialize_weights)
D.apply(initialize_weights)
criterion = nn.BCEWithLogitsLoss()
opt_G = torch.optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))
fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)
start_epoch = 0
history = {'G': [], 'D': []}

# Check shapes without changing BatchNorm running statistics.
G.eval(); D.eval()
with torch.no_grad():
    trial = G(fixed_noise[:2])
    assert trial.shape == (2, 3, 64, 64)
    assert D(trial).shape == (2,)
G.train(); D.train()
print('Network shape checks passed.')


## 4. Optional: resume a saved run
Skip on the first run. After a runtime reset, re-extract the dataset, run cells 1–3, then set `RESUME = True`. Use only checkpoints you trust. Restore both networks and optimizers, not just G. Epoch-boundary resuming is supported, but not an exact replay of random-number state or a partially completed epoch.


In [ ]:
RESUME = False
checkpoint_path = OUTPUT_DIR / 'latest.pt'
if RESUME:
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    if checkpoint['latent_dim'] != LATENT_DIM:
        raise ValueError('LATENT_DIM must match the saved run.')
    G.load_state_dict(checkpoint['G'])
    D.load_state_dict(checkpoint['D'])
    opt_G.load_state_dict(checkpoint['opt_G'])
    opt_D.load_state_dict(checkpoint['opt_D'])
    fixed_noise = checkpoint['fixed_noise'].to(device)
    history = checkpoint['history']
    start_epoch = checkpoint['epoch']
    print('Completed epochs:', start_epoch)


## 5. Train
Each batch takes two steps:
1. Teach D: real images should score as **1**, generated images as **0**. `detach()` prevents this step from updating G.
2. Teach G: generated images should fool D into predicting **1**. Gradients must flow through D to G here, so do NOT detach.

Fixed noise lets you compare the same samples over time. At each epoch end, save a grid and a checkpoint to Drive. Interrupting mid-epoch loses work since the last saved epoch.


In [ ]:
G.train(); D.train()
for epoch in range(start_epoch, EPOCHS):
    total_G = total_D = 0.0
    for step, real in enumerate(loader):
        real = real.to(device, non_blocking=True)
        b = real.size(0)
        ones = torch.ones(b, device=device)
        zeros = torch.zeros(b, device=device)

        # D update: distinguish real images from detached generated images.
        D.requires_grad_(True)
        opt_D.zero_grad(set_to_none=True)
        noise = torch.randn(b, LATENT_DIM, 1, 1, device=device)
        fake = G(noise)
        loss_D = criterion(D(real), ones) + criterion(D(fake.detach()), zeros)
        loss_D.backward()
        opt_D.step()
        opt_D.zero_grad(set_to_none=True)

        # G update: fool D. Freeze D parameters, not the computation graph.
        # Eval mode also keeps D's BatchNorm running statistics fixed here.
        D.requires_grad_(False)
        D.eval()
        opt_G.zero_grad(set_to_none=True)
        loss_G = criterion(D(fake), ones)
        loss_G.backward()
        opt_G.step()
        D.train()
        D.requires_grad_(True)

        total_G += loss_G.item()
        total_D += loss_D.item()
        if step % 100 == 0:
            print(f'Epoch {epoch+1}/{EPOCHS} | Batch {step+1}/{len(loader)} '
                  f'| D {loss_D.item():.3f} | G {loss_G.item():.3f}')

    history['G'].append(total_G / len(loader))
    history['D'].append(total_D / len(loader))
    G.eval()
    with torch.no_grad():
        samples = G(fixed_noise).cpu()
    save_tensor_image(samples, OUTPUT_DIR / f'epoch_{epoch+1:03d}.png',
                      nrow=8, normalize=True, value_range=(-1,1))
    G.train()
    state = dict(epoch=epoch+1, latent_dim=LATENT_DIM,
                 G=G.state_dict(), D=D.state_dict(),
                 opt_G=opt_G.state_dict(), opt_D=opt_D.state_dict(),
                 fixed_noise=fixed_noise.cpu(), history=history)
    # Write a temporary checkpoint first to reduce partial-file risk.
    temporary_path = OUTPUT_DIR / 'latest.tmp.pt'
    torch.save(state, temporary_path)
    temporary_path.replace(checkpoint_path)
    plt.figure(figsize=(8,8))
    plt.imshow(make_grid(samples, nrow=8, normalize=True, value_range=(-1,1)).permute(1,2,0))
    plt.title(f'Generated images — epoch {epoch+1}')
    plt.axis('off'); plt.show()
print('Saved outputs in:', OUTPUT_DIR)


## 6. Generate new images
This uses the generator's current weights. Running before training produces noise. For a later session, initialize the models and load the checkpoint first.


In [ ]:
G.eval()
with torch.no_grad():
    new_images = G(torch.randn(16, LATENT_DIM, 1, 1, device=device)).cpu()
save_tensor_image(new_images, OUTPUT_DIR / 'new_faces.png', nrow=4,
                  normalize=True, value_range=(-1,1))
plt.figure(figsize=(6,6))
plt.imshow(make_grid(new_images, nrow=4, normalize=True, value_range=(-1,1)).permute(1,2,0))
plt.axis('off'); plt.show()

plt.plot(history['G'], label='Generator')
plt.plot(history['D'], label='Discriminator (sum of real + fake losses)')
plt.xlabel('Completed epoch index (zero-based)'); plt.ylabel('Average loss')
plt.legend(); plt.show()


## What to expect and troubleshoot
- Early samples are noise. Face-like structure may emerge with more training and data; results depend on hardware, settings and training stability. This is not a text-to-image model.
- GAN losses oscillate; low loss is not proof of good output. Inspect fixed-noise grids for progress and diversity. Repeated nearly identical faces can indicate **mode collapse**.
- Missing images: your original unzip destination and your Drive image path differ. Point `IMAGE_DIR` at the actual folder of JPG files.
- Out of GPU memory: reduce `BATCH_SIZE` to 32 or 16 and rerun setup/model cells for a fresh run.
- Worker errors: try `num_workers=0`. Empty dataset: check paths before training.
- To continue: increase `EPOCHS` to a larger total target; load `latest.pt` through the optional resume cell. Do not rerun model initialization and accidentally overwrite a trained run without resuming.
- For quantitative evaluation later, use a proper held-out dataset and suitable generative metrics such as FID; a discriminator score is not a quality metric.
